# 15 FinBERT and Expanded Data Sanity Checks


## Files Checked


In [ ]:
from pathlib import Path
import pandas as pd
import re

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

processed = PROJECT_ROOT / 'data' / 'processed'
tables = PROJECT_ROOT / 'outputs' / 'tables'
suffix = '2022_2023_google_quality'

news = pd.read_csv(processed / f'news_target_tickers_{suffix}.csv')
scored = pd.read_csv(processed / f'news_target_tickers_finbert_scored_{suffix}.csv')
article_scores = pd.read_csv(processed / f'finbert_article_scores_{suffix}.csv')
daily = pd.read_csv(processed / f'daily_finbert_sentiment_features_{suffix}.csv')
model = pd.read_csv(processed / f'model_dataset_finbert_complete_{suffix}.csv')

## Coverage Checks


In [ ]:
def date_range(df, col):
    s = df[col].dropna().astype(str)
    return s.min(), s.max(), s.nunique()

coverage = {
    'news_published_utc': date_range(news, 'published_utc'),
    'scored_published_utc': date_range(scored, 'published_utc'),
    'aligned_trading_date': date_range(scored, 'aligned_trading_date'),
    'daily_sentiment_date': date_range(daily, 'date'),
    'model_trading_date': date_range(model, 'trading_date'),
}
coverage

## FinBERT Probability Checks


In [ ]:
probs = scored[['finbert_positive', 'finbert_negative', 'finbert_neutral']]
checks = {
    'rows': len(scored),
    'prob_sum_max_abs_error': float((probs.sum(axis=1) - 1).abs().max()),
    'label_mismatch_count': int((probs.idxmax(axis=1).str.replace('finbert_', '', regex=False) != scored['finbert_predicted_label']).sum()),
    'missing_probability_cells': int(probs.isna().sum().sum()),
    'duplicate_article_id_ticker_rows': int(scored.duplicated(['article_id', 'ticker']).sum()),
}
checks

In [ ]:
label_counts = scored['finbert_predicted_label'].value_counts().rename('count').to_frame()
label_counts['share_pct'] = (scored['finbert_predicted_label'].value_counts(normalize=True) * 100).round(1)
label_counts

## Text Quality Checks


In [ ]:
text = scored['text'].fillna('').astype(str)

def likely_duplicate_headline(s):
    first = s.split('. ', 1)[0].strip()
    rest = s.split('. ', 1)[1] if '. ' in s else ''
    return len(first) > 20 and first[:40].lower() in rest.lower()

text_quality = {
    'likely_duplicate_headline_rows': int(text.map(likely_duplicate_headline).sum()),
    'min_text_length': int(text.str.len().min()),
    'median_text_length': float(text.str.len().median()),
    'max_text_length': int(text.str.len().max()),
}
for pattern in ['â', 'Ã', 'Â', '€']:
    text_quality[f'encoding_artifact_rows_{pattern}'] = int(text.str.contains(re.escape(pattern), regex=True).sum())
text_quality

## Timestamp and Market-Close Checks


In [ ]:
pd.read_csv(tables / f'finbert_alignment_summary_{suffix}.csv')

## News Sufficiency


In [ ]:
pd.read_csv(tables / f'finbert_daily_coverage_{suffix}.csv').sort_values('total_articles', ascending=False)